In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("PySparkNotebookTest") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print(sys.executable)
print(spark)

C:\Users\USER\Documents\GitHub\pyspark-stock-analysis\.venv\Scripts\python.exe


In [3]:
data = [
    ("2024-01-01", 72000, 100000),
    ("2024-01-02", 73000, 120000),
    ("2024-01-03", 71000, 90000),
]

columns = ["date", "close", "volume"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+-----+------+
|      date|close|volume|
+----------+-----+------+
|2024-01-01|72000|100000|
|2024-01-02|73000|120000|
|2024-01-03|71000| 90000|
+----------+-----+------+



In [4]:
df.select("date","close").show()

+----------+-----+
|      date|close|
+----------+-----+
|2024-01-01|72000|
|2024-01-02|73000|
|2024-01-03|71000|
+----------+-----+



In [5]:
df.filter(df.close > 72000).show()

+----------+-----+------+
|      date|close|volume|
+----------+-----+------+
|2024-01-02|73000|120000|
+----------+-----+------+



In [6]:
from pyspark.sql.functions import col

df2 = df.withColumn("value", col("close") * col("volume"))

df2.show()

+----------+-----+------+----------+
|      date|close|volume|     value|
+----------+-----+------+----------+
|2024-01-01|72000|100000|7200000000|
|2024-01-02|73000|120000|8760000000|
|2024-01-03|71000| 90000|6390000000|
+----------+-----+------+----------+



In [7]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- close: long (nullable = true)
 |-- volume: long (nullable = true)



In [8]:
df.orderBy("close").show()

+----------+-----+------+
|      date|close|volume|
+----------+-----+------+
|2024-01-03|71000| 90000|
|2024-01-01|72000|100000|
|2024-01-02|73000|120000|
+----------+-----+------+



In [9]:
df.orderBy(col("close").desc()).show()

+----------+-----+------+
|      date|close|volume|
+----------+-----+------+
|2024-01-02|73000|120000|
|2024-01-01|72000|100000|
|2024-01-03|71000| 90000|
+----------+-----+------+



In [10]:
data2=[
("Samsung",72000),
("Samsung",73000),
("Apple",180),
("Apple",185),
("Tesla",240),
("Tesla",250)
]

df_company=spark.createDataFrame(data2,["company","price"])

In [11]:
df_company.groupBy("company").avg().show()

+-------+----------+
|company|avg(price)|
+-------+----------+
|Samsung|   72500.0|
|  Tesla|     245.0|
|  Apple|     182.5|
+-------+----------+



In [12]:
from pyspark.sql.functions import avg,max,min

df_company.groupBy("company").agg(
avg("price").alias("avg_price"),
max("price").alias("max_price"),
min("price").alias("min_price")
).show()

+-------+---------+---------+---------+
|company|avg_price|max_price|min_price|
+-------+---------+---------+---------+
|Samsung|  72500.0|    73000|    72000|
|  Tesla|    245.0|      250|      240|
|  Apple|    182.5|      185|      180|
+-------+---------+---------+---------+



In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

In [14]:
window=Window.orderBy("date").rowsBetween(-2,0)

In [15]:
df=df.withColumn("MA3",avg("close").over(window))

df.show()

+----------+-----+------+-------+
|      date|close|volume|    MA3|
+----------+-----+------+-------+
|2024-01-01|72000|100000|72000.0|
|2024-01-02|73000|120000|72500.0|
|2024-01-03|71000| 90000|72000.0|
+----------+-----+------+-------+

